In [2]:
import pandas as pd

# 1. Leer las bases
res = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital.xlsx")
scr18 = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scraper_2018_sin_dups.xlsx")
scr22 = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scraper_2022_sin_dups.xlsx")



In [15]:
# 2. Nos quedamos solo con ALCALDE DISTRITAL en los scrapers
scr18_alc = scr18[scr18["cargo"] == "ALCALDE DISTRITAL"].copy()
scr22_alc = scr22[scr22["cargo"] == "ALCALDE DISTRITAL"].copy()

# 3. Construimos el nombre completo del candidato
for df in (scr18_alc, scr22_alc):
    df["candidato_nombre_completo"] = (
        df["nombres"].astype(str).str.strip() + " " +
        df["apellidos"].astype(str).str.strip()
    )


In [16]:
# 4. Normalizamos llaves de unión (ubigeo y organización política)
def prep_claves(df, col_ubigeo="ubigeo", col_org="organizacion_politica"):
    df = df.copy()
    df["ubigeo_key"] = df[col_ubigeo].astype(str).str.strip()
    df["org_key"] = df[col_org].astype(str).str.upper().str.strip()
    return df

res = prep_claves(res, "ubigeo", "organizacion_politica")
scr18_alc = prep_claves(scr18_alc, "ubigeo", "organizacion_politica")
scr22_alc = prep_claves(scr22_alc, "ubigeo", "organizacion_politica")


In [17]:
# 5. Nos quedamos solo con las columnas necesarias de los scrapers
scr18_merge = scr18_alc[["ubigeo_key", "org_key", "dni", "candidato_nombre_completo"]]
scr22_merge = scr22_alc[["ubigeo_key", "org_key", "dni", "candidato_nombre_completo"]]

# 6. Separamos resultados por año
res_2018 = res[res["año"] == 2018].copy()
res_2022 = res[res["año"] == 2022].copy()


In [18]:
# 7. Unimos:
#    - 2018 con scraper 2018
#    - 2022 con scraper 2022
res_2018_merged = res_2018.merge(
    scr18_merge,
    how="left",
    on=["ubigeo_key", "org_key"],
    validate="m:1"   # opcional: asegura que en el scraper haya solo un alcalde por combinación
)

res_2022_merged = res_2022.merge(
    scr22_merge,
    how="left",
    on=["ubigeo_key", "org_key"],
    validate="m:1"
)


In [19]:
# 8. Volvemos a juntar 2018 + 2022
resultados_final = pd.concat([res_2018_merged, res_2022_merged], ignore_index=True)

# 9. (Opcional) Eliminamos columnas auxiliares de claves
resultados_final = resultados_final.drop(columns=["ubigeo_key", "org_key"])


In [20]:
# 10. Guardamos el resultado
resultados_final.to_excel("resultados_distrital_con_candidatos.xlsx", index=False)
print("Archivo generado: resultados_distrital_con_candidatos.xlsx")

Archivo generado: resultados_distrital_con_candidatos.xlsx


In [23]:
import pandas as pd
# 1. Cargar la base generada anteriormente
df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_con_candidatos.xlsx")

# 2. Asegurar que la variable de votos sea numérica
df["total_votos"] = pd.to_numeric(df["total_votos"], errors="coerce")


In [24]:
# 3. Ordenamos por distrito (ubigeo + año) y votos descendentes
df = df.sort_values(["ubigeo", "año", "total_votos"], ascending=[True, True, False])

# 4. Crear variable ganador:
#    1 si es el candidato con más votos en ese distrito-año
#    0 para los demás
df["ganador"] = df.groupby(["ubigeo", "año"])["total_votos"].transform(
    lambda x: (x == x.max()).astype(int)
)

In [1]:
# 5. Guardar la base actualizada
df.to_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_con_ganador.xlsx", index=False)

print("Archivo generado: resultados_distrital_con_ganador.xlsx")

NameError: name 'df' is not defined

In [18]:
import pandas as pd

# 1. Cargar la base de turnover 2022
turnover = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_turnover_2022.xlsx")

# 2. Cargar la base colapso4
colapso = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/colapso_con_ubigeo_reniec/colapso4.xlsx")


In [19]:
# 3. Asegurar que ubigeo esté en formato texto
turnover["ubigeo"] = turnover["ubigeo"].astype(str).str.strip()
colapso["ubigeo"] = colapso["ubigeo"].astype(str).str.strip()

# 4. Filtrar solo los años 2023–2025
colapso_2325 = colapso[colapso["año"].isin([2023, 2024, 2025])].copy()

# 5. Calcular la SUMA total de delitos por distrito (2023–2025)
delitos_suma = (
    colapso_2325.groupby("ubigeo")["cantidad"]
    .sum()
    .reset_index()
    .rename(columns={"cantidad": "delitos_suma_2023_2025"})
)

# 6. Merge con la base turnover 2022
final = turnover.merge(delitos_suma, on="ubigeo", how="left")



In [20]:
    
# 7. Guardar la base final
final.to_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_turnover_2022_con_delitos.xlsx", index=False)

print("Archivo generado: resultados_turnover_2022_con_delitos.xlsx")

Archivo generado: resultados_turnover_2022_con_delitos.xlsx
